## 1. Importing Necessary Libraries
This initial cell imports all the required Python libraries for the script. This includes os for file system operations, scipy for loading data from MATLAB's .mat files, matplotlib for plotting, numpy for numerical operations, and several key components from the pynwb library for creating and managing the Neurodata Without Borders (NWB) file.

### Attention:
Bear in mind that once nwb objects (unit data or behavior) are defined they cannot be modified and will cause errors if attempting to re-run the cell. Thus I advice restarting the kernel and running the notebook from the top,

In [1]:
import os
import scipy
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb import NWBFile, NWBHDF5IO
from datetime import datetime
import pytz
from pynwb.ecephys import ElectrodeGroup

## 2. Defining File Paths and Session Names
Useful if needed to process all sessions at once. 

In [2]:
directory = r"\\research-cifs.nyumc.org\research\buzsakilab\Homes\voerom01\NeuroNexus_Synapse"

names = ['NN_syn_20230601','NN_Syn_20230607','NN_syn_20230504']

## 3. Loading, Processing, and Organizing Data
This cell will load all data using cell explorer format and organize the variables we want to export as nwb

In [3]:
name = names[2]

print(name)

os.chdir(directory)

if name == 'NN_syn_20230504':
        os.chdir(r"\\research-cifs.nyumc.org\research\buzsakilab\Homes\voerom01\NeuroNexus_Synapse\NN_syn_20230504\8shanks\Hippo\NN_syn_20230504_HPC")
        cell_info = scipy.io.loadmat(name+'_HPC'+'.cell_metrics.cellinfo.mat',simplify_cells=True)['cell_metrics']
        ripple_events = scipy.io.loadmat(name+'_HPC.ripples.events.mat',simplify_cells=True)['ripples']

else:
    os.chdir(directory+"\\"+name)
    cell_info = scipy.io.loadmat(name+'.cell_metrics.cellinfo.mat',simplify_cells=True)['cell_metrics']
    ripple_events = scipy.io.loadmat(name+'.ripples.events.mat',simplify_cells=True)['ripples']

#
region = cell_info['tags']

if 'Good' in region.keys():
    good_cells = region['Good']
    
else:
    good_cells = cell_info['cellID']


good_cells = good_cells-1

ca1_units = region['CA1']
ca2_units = region['CA2']
ca3_units = region['CA3']
dg_units = region['DG']
cortex = region['cortex']


# 1. Combine all your lists into one to find the total range
all_ids = list(ca1_units) + list(ca2_units) + list(ca3_units) + list(dg_units) + list(cortex)
max_id = max(all_ids)

# 2. Create a list of 'None' or empty strings to act as a placeholder
# We use max_id + 1 in case your IDs are 1-indexed (like unit #10)
unit_labels = [None] * (max_id + 1)

# 3. Use your variables to "stamp" the region names at the correct indices
for i in ca1_units: unit_labels[i] = 'CA1'
for i in ca2_units: unit_labels[i] = 'CA2'
for i in ca3_units: unit_labels[i] = 'CA3'
for i in dg_units:  unit_labels[i] = 'DG'
for i in cortex:    unit_labels[i] = 'cortex'

# 4. Remove the None values if your IDs started at 1 or have gaps
cell_area = [label for label in unit_labels if label is not None]

len(cell_area)

cell_area = np.array(cell_area)[good_cells]

cell_types = cell_info['putativeCellType'][good_cells]
spike_times_all = cell_info["spikes"]['times'][good_cells]
firing_rates = cell_info['firingRate'][good_cells]
ab_ratio = cell_info['ab_ratio'][good_cells]
acg = cell_info['acg']['wide'][good_cells]
acg_t = cell_info['acg']['narrow'].T
acg = acg_t[good_cells]
burstIndex_Mizuseki2012 = cell_info['burstIndex_Mizuseki2012'][good_cells]
cv2 = cell_info['cv2'][good_cells]
maxWaveformCh = cell_info['maxWaveformCh'][good_cells]
troughToPeak = cell_info['troughToPeak'][good_cells]
waveforms = cell_info['waveforms']['raw'][good_cells]
acg_tau_decay = cell_info['acg_tau_decay'][good_cells]
acg_tau_rise = cell_info['acg_tau_rise'][good_cells]
thetaModulationIndex = cell_info['thetaModulationIndex'][good_cells]
x_position_probe = cell_info['trilat_x'][good_cells]
y_position_probe = cell_info['trilat_x'][good_cells]

metrics_2_add = [spike_times_all,cell_types,cell_area,firing_rates,ab_ratio,acg,burstIndex_Mizuseki2012,cv2,maxWaveformCh,troughToPeak,waveforms,acg_tau_decay,acg_tau_rise,thetaModulationIndex, x_position_probe, y_position_probe]


ripple_times = ripple_events['timestamps']
lfp_ripple_channel = ripple_events['detectorinfo']['detectionparms']['lfp']
srate_lfp = 1250
lfp_timestamps = np.arange(0,lfp_ripple_channel.shape[0]/srate_lfp,1/srate_lfp)

NN_syn_20230504


## 4. Creating the NWB File Object
This is the first step in building the NWB file. An NWBFile object is instantiated, serving as the main container for all experimental data and metadata. Essential metadata, such as the session description, start time, experimenter, and lab, are provided here.

In [4]:
# 1. Create NWBFile (Metadata is crucial)
session_start_time = datetime(2023, 5, 4, 10, 0, 0, tzinfo=pytz.utc)
nwbfile = NWBFile(
    session_description='Head-fix SiNAPS Recording',
    identifier='name',
    session_start_time=session_start_time,
    experimenter='Mihály Vöröslakos',
    lab='Buzsáki Lab',
    institution='NYU',
    # Add other required fields...
)

from pynwb.file import Subject
# 2. Add Subject metadata directly
nwbfile.subject = Subject(
    subject_id='NN_syn_M02',
    species='Mus musculus',
    strain='C57BL/6J',
    sex='M',
    age='P8W/P15W',      # Estimated age range for 21g female
    #weight='0.021 kg',
    description='Wild-type mouse'
)

## 5. Creating the Behavior Processing Module
NWB uses "processing modules" to organize derived data. This cell creates a module named behavior to group all processed behavioral data, such as the animal's position and speed, which will be added in the following steps.

## 7. Adding Spike Data and Cell Metrics to the Units Table

This is a critical step where all the spike data and associated cell metrics are added to the NWB file. The process involves three main parts:

Defining Hardware Metadata: An ElectrodeGroup is created to provide context about the recording probe.
Defining the Units Table Structure: Custom columns are added to the NWB file's units table using add_unit_column. Each column is given a name and a description, defining the structure that will hold all the detailed metrics for each neuron.
Populating the Table: The code iterates through each neuron, adding a new row to the units table using add_unit(). Each row is populated with the neuron's spike times and all of its corresponding metrics (cell type, firing rate, waveform shape, etc.).

In [5]:
# 2. Add an Electrodes table (essential for units)
# You would get this info from your cell_metrics or a separate file

#nwbfile.add_electrode_group(
#    name='ElectrodeGroup1',
#    description='45 degree insertion targetting hippocampus and neocortex',
#    location='CA3, CA1, RSC',
#    device=device
#)

device = nwbfile.create_device(name='SiNAPS')

e_group = ElectrodeGroup(
    name='ElectrodeGroup1',
    description='SiNAPS Probe targetting hippocampus',
    location='DG, CA3, CA1, CA1, cortex above probe (not RSC)',
    device=device  # Pass the device object here
)
nwbfile.add_electrode_group(e_group)

# --- ADD THIS SECTION BEFORE THE LOOP ---

# A. Non-Waveform Metrics (simple scalars or 1D arrays)
nwbfile.add_unit_column(name='cell_type', description='Cell classification (e.g., Pyramidal, Narrow Interneuron, Wide Interneuron)')
nwbfile.add_unit_column(name='cell_area', description='The brain region or subfield the cell was assigned to (e.g., CA1, CA3, RSC)')
nwbfile.add_unit_column(name='firing_rate', description='Firing rate in Hz: Spike count normalized by the interval between the first and the last spike..')
nwbfile.add_unit_column(name='ab_ratio', description='Waveform asymmetry; the ratio between the two positive peaks (peakB-peakA)/(peakA+peakB).')
nwbfile.add_unit_column(name='burstIndex_Mizuseki2012', description='Burst index as defined by Mizuseki et al. 2012.')
nwbfile.add_unit_column(name='cv2', description='Coefficient of variation (CV_2, 10.1152/jn.1996.75.5.1806).')
nwbfile.add_unit_column(name='maxWaveformCh', description='Max channel zero-indexed: The channel with the largest amplitude.')
nwbfile.add_unit_column(name='troughToPeak', description='Trough-to-peak latency is defined from the trough to the following peak of the waveform.')
nwbfile.add_unit_column(name='acg_tau_decay', description='Decay constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='acg_tau_rise', description='Rise constant (tau) of the ACG fit.')
nwbfile.add_unit_column(name='thetaModulationIndex', description='Theta modulation index. Originally defined in Cacucci et al., JNeuro 2004. Computed as the difference between the theta modulation trough (defined as mean of autocorrelogram bins, 50-70 msec) and the theta modulation peak (mean of autocorrelogram bins, 100-140 msec) over their sum, scaled from -1 to 1.')
nwbfile.add_unit_column(name='x_position_probe', description='Position along the x-axis for the max amp channel for each cell')
nwbfile.add_unit_column(name='y_position_probe', description='Position along the y-axis for the max amp channel for each cell')

# B. Complex Data (Waveforms and ACG)
# These are typically 1D arrays per unit, so we set the dtype to 'object' 
# to allow arrays of variable length/content (like NumPy arrays) to be stored in the column.
#nwbfile.add_unit_column(name='waveforms', description='Average raw spike waveform from channel with max amplitude.', dtype='object')
#nwbfile.add_unit_column(name='acg', description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', dtype='object')

# 1. WAVEFORMS
nwbfile.add_unit_column(
    name='waveforms', 
    description='Average raw spike waveform from channel with max amplitude.', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# 2. ACG
nwbfile.add_unit_column(
    name='acg', 
    description='Autocorrelogram (ACG) of spike times (wide [-1000 ms : 1 ms: 1000 ms]).', 
    index=True,
    data=np.array([], dtype='float32').reshape(0, 1) # Initialize with empty data array
)

# --- MODIFIED LOOP SECTION ---
# 3. Add Units table data
# Assuming cell_metrics is the original dict (used for firing_rate)
# and your new metrics are lists/arrays indexed by unit_i

for unit_i in range(len(spike_times_all)):
    # Get spike times (must be a 1D NumPy array in seconds)
    spike_times_list = spike_times_all[unit_i]
    
    nwbfile.add_unit(
        # Required arguments
        spike_times=spike_times_list,
        id=unit_i + 1,  # Unit IDs start at 1
        
        # --- ADDING YOUR METRICS ---
        
        # Scalar Metrics (from your list 'metrics_2_add')
        cell_type=cell_types[unit_i],
        cell_area=cell_area[unit_i],
        # Note: 'firing_rate' already existed in the original example, 
        # so we keep that structure if possible:
        firing_rate=firing_rates[unit_i], 
        ab_ratio=ab_ratio[unit_i],
        burstIndex_Mizuseki2012=burstIndex_Mizuseki2012[unit_i],
        cv2=cv2[unit_i],
        maxWaveformCh=maxWaveformCh[unit_i],
        troughToPeak=troughToPeak[unit_i],
        acg_tau_decay=acg_tau_decay[unit_i],
        acg_tau_rise=acg_tau_rise[unit_i],
        thetaModulationIndex=thetaModulationIndex[unit_i],
        x_position_probe=x_position_probe[unit_i],
        y_position_probe=y_position_probe[unit_i],

        # Array Metrics (WAVEFORMS and ACG)
        waveforms=waveforms[unit_i][:,np.newaxis],  # Must be a 1D or 2D NumPy array
        acg=acg[unit_i][:,np.newaxis] # Must be a 1D NumPy array
    )

C:\Users\GONZAJ81\AppData\Local\anaconda3\Lib\site-packages\pynwb\file.py:719: UserWarning: Column 'waveforms' is predefined in Units with index=2 which does not match the entered index argument. The predefined index spec will be ignored. Please ensure the new column complies with the spec. This will raise an error in a future version of HDMF.
  self.units.add_column(**kwargs)


In [6]:
# --- Assume these variables are loaded ---
# lfp_data_best_channel: 1D NumPy array of the best LFP trace (shape: time_points,)
# lfp_timestamps: 1D NumPy array of time points (in seconds)
# best_lfp_channel_id: Integer (e.g., 256) - the zero-indexed channel ID with max ripple power
# ----------------------------------------

from pynwb.ecephys import LFP, ElectricalSeries

# 1. Retrieve the existing electrode table
electrode_table = nwbfile.electrodes
# 2. Select the single electrode used for this LFP trace
lfp_channel_indices = np.array([0]) 

# Assuming 'e_group' is already defined from your code block.
import numpy as np

# Set the total number of channels on your probe (e.g., Neuropixels 2.0 = 384)
MAX_PROBE_CHANNELS = 384 

# Get the electrode table object (Correct way, fixing previous error)
electrode_table = nwbfile.electrodes 

# Check if electrodes have already been added (to prevent duplicate additions)
for channel_id in range(MAX_PROBE_CHANNELS):
    nwbfile.add_electrode(
        id=channel_id,
        x=np.nan,  # Placeholder if X/Y/Z are unknown
        y=np.nan,
        z=np.nan,
        imp=np.nan,
        location='CA1',
        filtering='low pass filter', # Standard filtering for spiking data
        group=e_group 
    )
print("Electrode table successfully populated.")


# Now, running the LFP creation code will work:
# electrode_table = nwbfile.electrodes # Already done above
# lfp_channel_indices = np.array([0]) # Or whatever your best channel ID is
# lfp_electrodes = nwbfile.create_electrode_table_region( ... ) # This will now execute

ecephys_module_name = 'ecephys'

ecephys_module = nwbfile.create_processing_module(
        name=ecephys_module_name, 
        description='Contains LFP data.'
    )

# Create the ElectrodeTableRegion (references the single best channel)
lfp_electrodes = nwbfile.create_electrode_table_region(
    region= [0],  # [best_lfp_channel_id]
    description=f'Single channel LFP selected based on highest ripple power.'
)

# 3. Create the ElectricalSeries
# NOTE: The data must be 2D (time_points, channels). Since you have 1 channel, use np.newaxis
lfp_electrical_series = ElectricalSeries(
    name='Best_Ripple_channel_LFP_CA1',
    data=lfp_ripple_channel[:, np.newaxis], # Convert 1D data to 2D (time_points, 1)
    electrodes=lfp_electrodes,
    starting_time=0.0, # Or the actual start time
    rate=1250.0,       # The sampling rate in Hz
    conversion=1e-6,
)

# 4. Add to the Ecephys Module (requires module creation/retrieval first, as shown previously)
# ... (Retrieve/Create ecephys_module) ...
lfp_container = LFP(electrical_series=lfp_electrical_series)
ecephys_module.add(lfp_container)

print(f"LFP from Ripple Channel added successfully.")

Electrode table successfully populated.
LFP from Ripple Channel added successfully.


C:\Users\GONZAJ81\AppData\Local\anaconda3\Lib\site-packages\hdmf\container.py:542: UserWarning: The linked table for DynamicTableRegion 'electrodes' does not share an ancestor with the DynamicTableRegion.
  child._validate_on_set_parent()


## 8. Saving all data into the .nwb file

In [7]:
os.chdir(r'C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695')
# 4. Write the file
with NWBHDF5IO(name+'.nwb', 'w') as io:
    io.write(nwbfile)

print("NWB file created successfully!")

NWB file created successfully!


## 9. Validate the .nwb file 
### This is mandatory for DANDI Upload 
If needed you should install DANDI packages

In [8]:
filename = name+".nwb"

# Use the $ symbol to inject the variable into the shell command
!dandi validate $filename

[DANDI.NON_DANDI_FILENAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\NN_syn_20230504.nwb — Filename does not conform to DANDI standard
[DANDI.NON_DANDI_FOLDERNAME] C:\Users\GONZAJ81\OneDrive - NYU Langone Health\Desktop\Subspace Paper Dataset\001695\NN_syn_20230504.nwb — File is not in folder at root with subject name


2026-01-13 21:48:35,674 [    INFO] Note: NumExpr detected 20 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
2026-01-13 21:48:35,675 [    INFO] NumExpr defaulting to 8 threads.
2026-01-13 21:48:37,687 [    INFO] Logs saved in C:\Users\GONZAJ81\AppData\Local\dandi\dandi-cli\Logs\2026.01.14-02.48.34Z-18308.log
